##### Copyright 2026 Google LLC.


In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Gemini API: Cross-modal Retrieval with Gemini Embedding 2 and Haystack


 <a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/haystack/Gemini_Embedding_Haystack_Crossmodal_Retrieval.ipynb"><img
  src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>


> **Note:** This notebook requires paid tier rate limits to run properly. (cf. [pricing](https://ai.google.dev/pricing) for more details).


## Overview


[Gemini Embedding 2](https://ai.google.dev/gemini-api/docs/models/gemini-embedding-2-preview) is a multimodal embedding model, that maps text, images, video, audio, and PDFs into a single unified embedding space.

[Haystack](https://haystack.deepset.ai/) is an open-source AI orchestration framework for building context-engineered, production-ready LLM applications. You can use it to build Agents, RAG, multimodal applications, semantic search and more.

In this notebook, you'll see several ways to use Gemini Embedding 2 with Haystack, starting with textual retrieval and RAG, and expanding to several cross-modal examples where you use one modality to search on others. You can discover more about this integration in the [launch blog post](https://haystack.deepset.ai/blog/multimodal-embeddings-gemini-haystack).


<!-- Community Contributor Badge -->
<table>
  <tr>
    <!-- Author Avatar Cell -->
    <td bgcolor="#d7e6ff">
      <a href="https://github.com/anakin87" target="_blank" title="View anakin87's profile on GitHub">
        <img src="https://github.com/anakin87.png?size=100"
             alt="anakin87's GitHub avatar"
             width="100"
             >
      </a>
    </td>
    <td bgcolor="#d7e6ff">
      <a href="https://github.com/bilgeyucel" target="_blank" title="View bilgeyucel's profile on GitHub">
        <img src="https://github.com/bilgeyucel.png?size=100"
             alt="bilgeyucel's GitHub avatar"
             width="100"
             >
      </a>
    </td>     
    <!-- Text Content Cell -->
    <td bgcolor="#d7e6ff">
      <h2><font color='black'>This notebook was contributed by <a href="https://github.com/anakin87" target="_blank"><font color='#217bfe'><strong>Stefano Fiorucci</strong></font></a> and <a href="https://github.com/bilgeyucel" target="_blank"><font color='#217bfe'><strong>Bilge Yücel</strong></font></a>.</font></h2>
      <h5><font color='black'><a href="https://www.linkedin.com/in/stefano-fiorucci"><font color="#078efb">Stefano's LinkedIn</font></a> - <a href="https://www.linkedin.com/in/bilge-yucel"><font color="#078efb">Bilge's LinkedIn</font></a></h5></font><br>
      <!-- Footer -->
      <font color='black'><small><em>Have a cool Gemini example? Feel free to <a href="https://github.com/google-gemini/cookbook/blob/main/CONTRIBUTING.md" target="_blank"><font color="#078efb">share it too</font></a>!</em></small></font>
    </td>
  </tr>
</table>


## Setup

First, install the packages and configure your API key.

### Installation

Install Haystack's Python library, `haystack-ai`, Haystack's integration package for the Gemini API, `google-genai-haystack`, and other libraries needed to run the examples.


In [ ]:
%pip install --quiet haystack-ai google-genai-haystack
%pip install --quiet datasets pypdfium2 pillow


### Configure your API key

To run the following cell, your API key must be stored in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for an example.


In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")


In [ ]:
EMBEDDING_MODEL = "gemini-embedding-2"  # @param ["gemini-embedding-2"] {"allow-input":true, isTemplate: true}
GENERATIVE_MODEL = "gemini-3.1-flash-lite-preview"  # @param ["gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}


## Textual Retrieval

In this first example, you will embed a collection of documents about the Seven Wonders of the Ancient World and then search over it using semantic similarity.

You use:
- [`GoogleGenAIDocumentEmbedder`](https://docs.haystack.deepset.ai/docs/googlegenaidocumentembedder) to embed documents containing text
- [`GoogleGenAITextEmbedder`](https://docs.haystack.deepset.ai/docs/googlegenaitextembedder) to embed a query (textual string)


### Create, embed, and store documents

Create the documents starting from a dataset stored on Hugging Face, embed them with `GoogleGenAIDocumentEmbedder` and then store them in the [`InMemoryDocumentStore`](https://docs.haystack.deepset.ai/docs/inmemorydocumentstore).

`InMemoryDocumentStore` is the simplest DocumentStore to get started with but it's not recommended for production. To learn more about the different types of external databases that Haystack supports, see [DocumentStore Integrations](https://haystack.deepset.ai/integrations?type=Document+Store).


In [ ]:
from datasets import load_dataset
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack_integrations.components.embedders.google_genai import (
    GoogleGenAIDocumentEmbedder,
    GoogleGenAITextEmbedder,
)

document_store = InMemoryDocumentStore(embedding_similarity_function="cosine")

dataset = load_dataset("bilgeyucel/seven-wonders", split="train")

docs = [Document(content=doc["content"], meta=doc["meta"]) for doc in dataset]

doc_embedder = GoogleGenAIDocumentEmbedder(model=EMBEDDING_MODEL, batch_size=32)
docs_with_embeddings = doc_embedder.run(docs)
document_store.write_documents(docs_with_embeddings["documents"])


### Retrieve documents

You can now embed the query and use [`InMemoryEmbeddingRetriever`](https://docs.haystack.deepset.ai/docs/inmemoryembeddingretriever) to look for similar documents.


In [ ]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

retriever = InMemoryEmbeddingRetriever(document_store)

query = "What does Statue of Zeus look like?"  # @param {type:"string"}
text_embedder = GoogleGenAITextEmbedder(model=EMBEDDING_MODEL)
query_embedding = text_embedder.run(query)["embedding"]

result = retriever.run(query_embedding=query_embedding, top_k=3)

for doc in result["documents"]:
    print(doc.meta)
    print(doc.content)
    print(doc.score)
    print("-" * 10)


{'url': 'https://en.wikipedia.org/wiki/Statue_of_Zeus_at_Olympia', '_split_id': 0}
The Statue of Zeus at Olympia was a giant seated figure, about 12.4 m (41 ft) tall,[1] made by the Greek sculptor Phidias around
 435 BC at the sanctuary of Olympia, Greece, and erected in the Temple of Zeus there.  Zeus is the sky and thunder god in ancien
t Greek religion, who rules as king of the gods of Mount Olympus.
The statue was a chryselephantine sculpture of ivory plates and gold panels on a wooden framework. Zeus sat on a painted cedarwo
od throne ornamented with ebony, ivory, gold, and precious stones. It was one of the Seven Wonders of the Ancient World.
The statue was lost and destroyed before the end of the 5th century AD, with conflicting accounts of the date and circumstances.
 Details of its form are known only from ancient Greek descriptions and representations on coins.
Coin from Elis district in southern Greece illustrating the Olympian Zeus statue (Nordisk familjebok)
History[edit]


### Create a RAG pipeline

Now that you have good retrieval results, you can create a RAG [Pipeline](https://docs.haystack.deepset.ai/docs/pipelines).

In addition to the components introduced above, you use:
- [ChatPromptBuilder](https://docs.haystack.deepset.ai/docs/chatpromptbuilder) to create dynamic prompts
- [GoogleGenAIChatGenerator](https://docs.haystack.deepset.ai/docs/googlegenaichatgenerator), the LLM, with `gemini-3.1-flash-lite-preview`

To understand better how Pipelines work on Haystack, take a look at the [ Creating Your First QA Pipeline with Retrieval-Augmentation tutorial](https://haystack.deepset.ai/tutorials/27_first_rag_pipeline).


In [ ]:
from haystack import Pipeline
from haystack.components.builders.chat_prompt_builder import ChatPromptBuilder
from haystack_integrations.components.generators.google_genai import GoogleGenAIChatGenerator

template = """
{% message role="user"%}
Given these documents, briefly answer the question.
Documents:
{% for doc in documents %}
    {{ doc.content }}
{% endfor %}

Question: {{ query }}
{% endmessage %}
"""

pipe = Pipeline()
pipe.add_component(
    "text_embedder",
    GoogleGenAITextEmbedder(
        model=EMBEDDING_MODEL, prefix = "task: search result | query: "
    ),
)
pipe.add_component("retriever", InMemoryEmbeddingRetriever(document_store))
pipe.add_component(
    "prompt_builder", ChatPromptBuilder(template=template, required_variables="*")
)
pipe.add_component("llm", GoogleGenAIChatGenerator(model=GENERATIVE_MODEL))

pipe.connect("text_embedder.embedding", "retriever.query_embedding")
pipe.connect("retriever", "prompt_builder")
pipe.connect("prompt_builder", "llm")

query = "What does Statue of Zeus look like?"  # @param {type:"string"}

results = pipe.run(
    {
        "text_embedder": {"text": query},
        "prompt_builder": {"query": query},
    },
)

print(results["llm"]["replies"][0].text)


The Statue of Zeus was a 12.4-meter (41-foot) tall chryselephantine (gold and ivory) seated figure. Key features included:
*   **Appearance:** Zeus was depicted with curly hair, often adorned with a sculpted wreath of olive sprays, and wore a gilded r
obe decorated with glass, lilies, and animals.
*   **Pose and Objects:** He sat on a painted cedarwood throne encrusted with ebony, ivory, gold, and precious stones. In his ri
ght hand, he held a small statue of the goddess Nike; in his left, he held a scepter topped with an eagle.
*   **Details:** His feet rested on a footstool featuring a relief of an Amazonomachy. Legend suggests the sculptor Phidias carv
ed "Pantarkes is beautiful" onto Zeus's little finger.
*   **Setting Effects:** The statue sat behind a reflecting pool of olive oil, which helped protect the ivory and doubled the st
atue's apparent height.
Because no original survives, these details are derived from ancient descriptions by writers like Pausanias and representations


## Cross-modal retrieval: Use text to search over multiple modalities

Since Gemini Embedding 2 embeds text, images, PDF, audio and videos in the same embedding space, it allows cross-modal retrieval.

In this example, you use a textual query to search over a collection of data in different modalities.


### Download files

Download some files from the web, mostly about animals.


In [ ]:
!wget -q -O kangaroo.mp4 "https://www.pexels.com/download/video/5739690/"
!wget -q -O tiger.jpg "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b0/Bengal_tiger_%28Panthera_tigris_tigris%29_female_3_crop.jpg/960px-Bengal_tiger_%28Panthera_tigris_tigris%29_female_3_crop.jpg"
!wget -q -O sample.pdf "https://pdfobject.com/pdf/sample.pdf"
!wget -q -O dog.jpg "https://upload.wikimedia.org/wikipedia/commons/3/3b/BlkStdSchnauzer2.jpg"
!wget -q -O cat.jpg "https://upload.wikimedia.org/wikipedia/commons/thumb/1/12/Tabby_cat_with_visible_nictitating_membrane.jpg/500px-Tabby_cat_with_visible_nictitating_membrane.jpg"


### Embed documents

Since you're not embedding text here, use the [`GoogleGenAIMultimodalDocumentEmbedder`](https://docs.haystack.deepset.ai/docs/googlegenaimultimodaldocumentembedder), along with documents only specifying the file path in their meta.


In [ ]:
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack import Document
from haystack_integrations.components.embedders.google_genai import GoogleGenAIMultimodalDocumentEmbedder

document_store = InMemoryDocumentStore()

docs = [
    Document(meta={"file_path": "kangaroo.mp4"}),
    Document(meta={"file_path": "tiger.jpg"}),
    Document(meta={"file_path": "sample.pdf"}),
    Document(meta={"file_path": "dog.jpg"}),
    Document(meta={"file_path": "cat.jpg"}),
]

doc_multimodal_embedder = GoogleGenAIMultimodalDocumentEmbedder(model=EMBEDDING_MODEL)
docs_with_embeddings = doc_multimodal_embedder.run(docs)
document_store.write_documents(docs_with_embeddings["documents"])


Calculating embeddings: 1it [00:02,  2.24s/it]


5


In [ ]:
document_store.filter_documents()


[Document(id=908c54133d4714ab9136e445fac4b0f12c5f56cb81964dc6e4dc751670fc4d4b, meta: {'file_path': 'kangaroo.mp4'}, embedding: v
ector of size 3072),
 Document(id=9bbed843bfb04758cc964a879950459c0ad09234b858dd88716c8a5d6596b76d, meta: {'file_path': 'tiger.jpg'}, embedding: vect
or of size 3072),
 Document(id=3d4d49a88ce134f79183d90cd9bcb8bdb3c3e9b0b054252f8a682c5c11c6ed2e, meta: {'file_path': 'sample.pdf'}, embedding: vec
tor of size 3072),
 Document(id=0c96d7a3ae6ac63fb815d8f4261fe9cd91acadc00e2f3aa18fe83ff5ae5bd855, meta: {'file_path': 'dog.jpg'}, embedding: vector
 of size 3072),
 Document(id=3977c4952d7062e7317401bc941fb0d080c81366ffcdeb5c9b579438f2d453c0, meta: {'file_path': 'cat.jpg'}, embedding: vector
 of size 3072)]


### Search for similar documents

You can now embed the query and use [`InMemoryEmbeddingRetriever`](https://docs.haystack.deepset.ai/docs/inmemoryembeddingretriever) to look for similar documents.


In [ ]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack_integrations.components.embedders.google_genai import GoogleGenAITextEmbedder


retriever = InMemoryEmbeddingRetriever(document_store)

query = "australian animals"  # @param {type:"string"}
text_embedder = GoogleGenAITextEmbedder(model=EMBEDDING_MODEL)
query_embedding = text_embedder.run(query)["embedding"]

result = retriever.run(query_embedding=query_embedding)

for doc in result["documents"]:
    print(doc.meta)
    print(doc.score)
    print("-" * 10)


{'file_path': 'kangaroo.mp4'}
0.4001252032536827
----------
{'file_path': 'tiger.jpg'}
0.3048929565421354
----------
{'file_path': 'cat.jpg'}
0.30093315040100627
----------
{'file_path': 'dog.jpg'}
0.29233270373478404
----------
{'file_path': 'sample.pdf'}
0.2344690758512073
----------


## Cross-modal retrieval: Search over PDF pages

Haystack makes it easy to convert a PDF document into JPEG images (one for each page) that can be embedded separately, for more precise retrieval.

In this example, you'll be using a textual query to search for a plot in the Gemma 3 Technical Report.


### Download the paper and embed it


In [ ]:
!wget -q "https://arxiv.org/pdf/2503.19786" -O gemma_3_report.pdf


In [ ]:
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore

from haystack_integrations.components.embedders.google_genai import (
    GoogleGenAIMultimodalDocumentEmbedder,
)

documents = [
    Document(meta={"file_path": "gemma_3_report.pdf", "page_number": i})
    for i in range(1, 12)
]

multimodal_document_embedder = GoogleGenAIMultimodalDocumentEmbedder(
    model=EMBEDDING_MODEL
)

result = multimodal_document_embedder.run(documents)

document_store = InMemoryDocumentStore()
document_store.write_documents(result["documents"])


Calculating embeddings: 2it [00:02,  1.27s/it]


11


### Search for relevant pages

You can now embed the query and use [`InMemoryEmbeddingRetriever`](https://docs.haystack.deepset.ai/docs/inmemoryembeddingretriever) to look for relevant pages.


In [ ]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack_integrations.components.embedders.google_genai import GoogleGenAITextEmbedder

query = "Evaluation of Gemma 3 27B IT model in the Chatbot Arena table"  # @param {type:"string"}

text_embedder = GoogleGenAITextEmbedder(model=EMBEDDING_MODEL)
query_embedding = text_embedder.run(text=query)["embedding"]
embedding_retriever = InMemoryEmbeddingRetriever(document_store=document_store)

results = embedding_retriever.run(query_embedding=query_embedding, top_k=3)["documents"]
for doc in results:
    print(doc.meta)
    print(doc.score)
    print("-" * 100)


{'file_path': 'gemma_3_report.pdf', 'page_number': 5}
0.52335352400811
----------------------------------------------------------------------------------------------------
{'file_path': 'gemma_3_report.pdf', 'page_number': 4}
0.4993533840882688
----------------------------------------------------------------------------------------------------
{'file_path': 'gemma_3_report.pdf', 'page_number': 6}
0.4431172369214042
----------------------------------------------------------------------------------------------------


### Verify retrieval

Check if the retrieved page is relevant, by displaying it.


In [ ]:
#@title Helper function to display a PDF page
import pypdfium2 as pdfium
from IPython.display import display


def show_pdf_page(file, page_number):
    pdf = pdfium.PdfDocument(file)
    page = pdf[page_number - 1]
    bitmap = page.render(scale=2)
    img = bitmap.to_pil()
    display(img)


In [ ]:
show_pdf_page("gemma_3_report.pdf", results[0].meta["page_number"])


## Cross-modal retrieval: Image to text search

In this example, you use an image embedding to retrieve similar texts.

Because all modalities share the same vector space, you can use this approach to support cross-modal retrieval in any direction, for example text-to-image, image-to-text, audio-to-video, or video-to-document search.

You also choose the embedding dimensionality: since Gemini embedding models are trained with Matryoshka Representation Learning (MRL), you can control the size of the output embedding, to save storage and increase computational efficiency, with minimal quality loss.


### Create a collection of documents, embed and store them


In [ ]:
from haystack import Document

documents = [
    Document(
        content=(
            "The capybara is the largest rodent in the world and is native to South America, "
            "where it lives near rivers, lakes, and wetlands. It is highly social and often "
            "seen relaxing in groups, spending much of its time swimming or soaking in water."
        )
    ),
    Document(
        content=(
            "Dogs are domesticated mammals known for their loyalty, intelligence, and strong "
            "bond with humans. They have been bred for thousands of years for roles such as "
            "companionship, hunting, guarding, and assisting people with various tasks."
        )
    ),
    Document(
        content=(
            "The tiger is the largest species of big cat and is recognized by its distinctive "
            "orange coat with black stripes. It is a powerful solitary predator that inhabits "
            "forests, grasslands, and wetlands across parts of Asia."
        )
    ),
    Document(
        content=(
            "The giraffe is the tallest land animal on Earth, easily identified by its long neck "
            "and distinctive spotted coat. It uses its height to reach leaves high in acacia trees "
            "and roams the savannas and open woodlands of Africa."
        )
    ),
]


In [ ]:
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack_integrations.components.embedders.google_genai import (
    GoogleGenAIDocumentEmbedder,
)

document_store = InMemoryDocumentStore(embedding_similarity_function="cosine")

doc_embedder = GoogleGenAIDocumentEmbedder(
    model=EMBEDDING_MODEL,
    batch_size=5,
    config={"output_dimensionality": 768},
)
docs_with_embeddings = doc_embedder.run(documents)
document_store.write_documents(docs_with_embeddings["documents"])


Calculating embeddings: 1it [00:00,  2.71it/s]


4


In [ ]:
document_store.filter_documents()


[Document(id=d65ac24c9a56b8928808cedc1ddd923348191213f318ba6d7aa4d31395f6d94d, content: 'The capybara is the largest rodent in t
he world and is native to South America, where it lives near ...', embedding: vector of size 768),
 Document(id=9a168b58aea0919914b4a385895aa1bfb8ee5f0500807d67921c36c8fc7af47d, content: 'Dogs are domesticated mammals known for
 their loyalty, intelligence, and strong bond with humans. Th...', embedding: vector of size 768),
 Document(id=930e364d624edc95158e502f6ba0525e3babe30be9b7c073515566c3280ece15, content: 'The tiger is the largest species of big
 cat and is recognized by its distinctive orange coat with bl...', embedding: vector of size 768),
 Document(id=027a82bad0779057322c26d6063d7189c34369abe270c7842eb8286e5ad921da, content: 'The giraffe is the tallest land animal
on Earth, easily identified by its long neck and distinctive ...', embedding: vector of size 768)]


### Download an image and embed it

Now, download a dog image to search for similar textual documents.


In [ ]:
!wget -q -O dog.jpg https://upload.wikimedia.org/wikipedia/commons/3/3b/BlkStdSchnauzer2.jpg


In [ ]:
from haystack_integrations.components.embedders.google_genai import (
    GoogleGenAIMultimodalDocumentEmbedder,
)

image_doc = Document(meta={"file_path": "dog.jpg"})

image_embedder = GoogleGenAIMultimodalDocumentEmbedder(
    model=EMBEDDING_MODEL, config={"output_dimensionality": 768}
)

image_embedding = image_embedder.run([image_doc])["documents"][0].embedding


Calculating embeddings: 1it [00:01,  1.12s/it]


In [ ]:
len(image_embedding)


768


### Use the image embedding to find relevant documents


In [ ]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

embedding_retriever = InMemoryEmbeddingRetriever(document_store=document_store)

results = embedding_retriever.run(query_embedding=image_embedding, top_k=3)["documents"]
for doc in results:
    print(doc.content)
    print(doc.score)
    print("-" * 100)


Dogs are domesticated mammals known for their loyalty, intelligence, and strong bond with humans. They have been bred for thousa
nds of years for roles such as companionship, hunting, guarding, and assisting people with various tasks.
0.3451460525453204
----------------------------------------------------------------------------------------------------
The capybara is the largest rodent in the world and is native to South America, where it lives near rivers, lakes, and wetlands.
 It is highly social and often seen relaxing in groups, spending much of its time swimming or soaking in water.
0.26969933652787453
----------------------------------------------------------------------------------------------------
The giraffe is the tallest land animal on Earth, easily identified by its long neck and distinctive spotted coat. It uses its he
ight to reach leaves high in acacia trees and roams the savannas and open woodlands of Africa.
0.2627500247245441
----------------------------------------

## What's next?

This notebook showed how to combine Gemini Embedding 2 with Haystack for textual retrieval, RAG, and cross-modal retrieval.

There are many more use cases for the Haystack-Gemini integration. You can explore them [here](https://haystack.deepset.ai/integrations/google-genai).
